# MyBizMate Scan — `PP-OCRv6_small_rec` (100-Class Latin Cut) Fine-Tuning Notebook

Self-contained Google Colab GPU pipeline that fine-tunes **PaddleOCR `PP-OCRv6_small_rec`** for **MyBizMate Sales Invoices, Service Invoices, and BIR Form 2307**, exports a drop-in **`inference_latin.onnx`**, verifies every C++ `recognizer.cpp` contract, and generates the updated `model.json`.

**Before running:** In Colab, select **Runtime → Change runtime type → T4 GPU** (or L4 / A100), then click **Runtime → Run all**.

In [ ]:
# === CELL 1: GPU CHECK, PADDLEPADDLE-GPU, PADDLEOCR & HANDWRITING FONTS ===
import os
import re
import subprocess
import urllib.request
from pathlib import Path

WORK_DIR = Path('/content/mybizmate_ocr_ft')
WORK_DIR.mkdir(parents=True, exist_ok=True)

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
assert smi.returncode == 0, 'No NVIDIA GPU detected! Set Runtime -> Change runtime type -> T4 GPU.'
print(smi.stdout.splitlines()[0])
m_cuda = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi.stdout)
cuda_major = int(m_cuda.group(1)) if m_cuda else 12

paddle_index = (
    'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
    if cuda_major >= 12
    else 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'
)
subprocess.check_call(['pip', 'install', '-q', 'paddlepaddle-gpu', '-i', paddle_index])
subprocess.check_call([
    'pip', 'install', '-q',
    'paddle2onnx', 'onnx', 'onnxruntime',
    'shapely', 'scikit-image', 'pyclipper', 'lmdb', 'tqdm',
    'rapidfuzz', 'opencv-python-headless', 'pillow', 'pyyaml', 'albumentations', 'albucore'
])

PADDLE_OCR_DIR = WORK_DIR / 'PaddleOCR'
if not PADDLE_OCR_DIR.exists():
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLE_OCR_DIR)
    ])

FONTS_DIR = WORK_DIR / 'fonts'
FONTS_DIR.mkdir(exist_ok=True)
GOOGLE_HANDWRITING_FONTS = [
    ('PatrickHand-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/patrickhand/PatrickHand-Regular.ttf'),
    ('Kalam-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/kalam/Kalam-Regular.ttf'),
    ('IndieFlower-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/indieflower/IndieFlower-Regular.ttf'),
    ('ShadowsIntoLight.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/shadowsintolight/ShadowsIntoLight.ttf'),
    ('ArchitectsDaughter-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/architectsdaughter/ArchitectsDaughter-Regular.ttf'),
    ('Handlee-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/handlee/Handlee-Regular.ttf'),
    ('ShortStack-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/shortstack/ShortStack-Regular.ttf'),
    ('Delius-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/delius/Delius-Regular.ttf'),
    ('Neucha.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/neucha/Neucha.ttf'),
    ('Courgette-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/courgette/Courgette-Regular.ttf'),
    ('Mali-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/mali/Mali-Regular.ttf'),
    ('Caveat-Variable.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/caveat/Caveat%5Bwght%5D.ttf'),
    ('ComicNeue-Regular.ttf', 'https://raw.githubusercontent.com/google/fonts/main/ofl/comicneue/ComicNeue-Regular.ttf'),
]
for font_name, font_url in GOOGLE_HANDWRITING_FONTS:
    dest = FONTS_DIR / font_name
    if not dest.exists():
        urllib.request.urlretrieve(font_url, dest)

import paddle
print('Paddle version:', paddle.__version__, '| Compiled with CUDA:', paddle.is_compiled_with_cuda())
print('Downloaded handwriting fonts:', len(list(FONTS_DIR.glob('*.ttf'))))

In [ ]:
# === CELL 2: WRITE EXACT 98-CHARACTER dict.txt (100 CLASSES WITH <blank> + SPACE) ===
DICT_LINES = [chr(c) for c in range(33, 127)] + ['É', 'Ñ', 'é', 'ñ']
assert len(DICT_LINES) == 98, f'Expected 98 chars, got {len(DICT_LINES)}'

DICT_PATH = WORK_DIR / 'dict.txt'
DICT_PATH.write_bytes('\n'.join(DICT_LINES).encode('utf-8'))
print('Wrote dict.txt ->', len(DICT_LINES), 'lines (+ <blank> at 0 + space at 99 = 100 classes)')

In [ ]:
# === CELL 3: GENERATE SYNTHETIC FILIPINO INVOICE & BIR 2307 HANDWRITING DATASET ===
# Optional: If you also upload MyBizMate Scan training data *.zip files into /content/exports/,
# this cell automatically merges them into train/val alongside synthetic replay crops!
import json
import random
import shutil
import zipfile
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

SYNTH_TRAIN_COUNT = 10000
SYNTH_VAL_COUNT = 2000
MAX_TEXT_LENGTH = 25
CROP_PADDING_PX = 6
SEED = 20261123

DATASET_DIR = Path('/content/rec_dataset')
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
(DATASET_DIR / 'train').mkdir(parents=True)
(DATASET_DIR / 'val').mkdir(parents=True)

ALLOWED_CHARS = set(DICT_LINES) | {' '}
ACCENTED_CHARS = ('É', 'Ñ', 'é', 'ñ')

UNITS = ('pc', 'pcs', 'pack', 'packs', 'box', 'tray', 'sack', 'kg', 'g', 'L', 'mL', 'can', 'btl', 'roll', 'set', 'doz', 'bag', 'ream')
ITEMS = (
    'Eggs', 'Sugar', 'Sardines', 'Bread Loaf', 'Alaska Evap 370ml', 'Nails 2in',
    'Rice 25kg', 'Plywood 1/4', 'Bear Brand 300g', 'Cement 40kg', 'Cooking Oil 1L',
    'Soy Sauce 385mL', 'Office supplies', 'Printing service', 'Delivery fee',
    'Store rental', 'Jalapeño Cheese', 'Piña Juice 1L', 'Café Barako', 'Éclair Box', 'Niño Notebook'
)
PARTIES_PLACES = (
    'Parañaque Hardware', 'Muñoz Trading', 'Zuñiga Mdse.', 'Sto. Niño Store',
    'Dasmariñas Cavite', 'Biñan Laguna', 'Peñafrancia Ave.', 'Doña Juana Subd.',
    'Bañez Construction', 'Castañeda Supply', 'Ibañez Auto', 'San José Café', 'Las Piñas'
)
ATC_CODES = (
    'WC010', 'WC100', 'WC120', 'WC140', 'WC158', 'WC160', 'WC610', 'WC640',
    'WI010', 'WI050', 'WI060', 'WI100', 'WI120', 'WI158', 'WI160', 'WB080', 'WB082'
)
MONTHS = ('Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec')

def font_supports_accents(fp: Path) -> bool:
    try:
        f = ImageFont.truetype(str(fp), size=32)
        return f.getmask('Ñ').size[0] > 0 and list(f.getmask('Ñ')) != list(f.getmask('\uffff'))
    except OSError:
        return False

def make_label(rng: random.Random) -> tuple[str, str]:
    for _ in range(20):
        r = rng.random()
        if r < 0.36:
            if rng.random() < 0.12:
                kind, s = 'amount', '0.00'
            else:
                val = rng.uniform(15.0, 480000.0)
                s = f'{val:,.2f}' if rng.random() < 0.65 else f'{int(round(val)):,}'
                if rng.random() < 0.35:
                    s = s.replace(',', '')
                if rng.random() < 0.18:
                    s = 'P' + s
                kind = 'amount'
        elif r < 0.48:
            kind = 'number'
            s = str(rng.randint(1, 200)) if rng.random() < 0.72 else f"{rng.randint(1, 9)} {rng.choice(('1/2', '1/4', '3/4'))}"
        elif r < 0.60:
            kind = 'date'
            m, d, y = rng.randint(1, 12), rng.randint(1, 28), rng.randint(2024, 2028)
            st = rng.randint(0, 3)
            s = (
                f'{MONTHS[m-1]} {d}, {y}' if st == 0 else
                f'{d} {MONTHS[m-1]} {y}' if st == 1 else
                f"{m:02d}{rng.choice(('-', '/'))}{d:02d}{rng.choice(('-', '/'))}{y % 100:02d}" if st == 2 else
                f'{m}/{d}/{y}'
            )
        elif r < 0.68:
            kind = 'digits'
            s = f'{rng.randint(1, 999999):07d}' if rng.random() < 0.6 else f'{rng.randint(1,12):02d}012026'
        elif r < 0.76:
            kind = 'tin'
            a, b, c = rng.randint(100, 999), rng.randint(0, 999), rng.randint(0, 999)
            s = f'{a}-{b:03d}-{c:03d}-0000' if rng.random() < 0.55 else f'{a}{b:03d}{c:03d}0000'
        elif r < 0.84:
            kind, s = 'code', rng.choice(ATC_CODES)
        elif r < 0.91:
            kind, s = 'unit', rng.choice(UNITS)
        else:
            kind, s = 'text', rng.choice(ITEMS + PARTIES_PLACES)
        if 1 <= len(s) <= MAX_TEXT_LENGTH and all(ch in ALLOWED_CHARS for ch in s):
            return kind, s
    return 'amount', '1,250.00'

def render_crop(label: str, font_path: Path, accent_fallback: Path | None, rng: random.Random) -> Image.Image:
    fp = font_path
    if any(ch in label for ch in ACCENTED_CHARS) and not font_supports_accents(fp) and accent_fallback:
        fp = accent_fallback
    fsize = rng.randint(26, 38)
    font = ImageFont.truetype(str(fp), size=fsize)
    p_base = rng.randint(233, 252)
    tint = rng.randint(-3, 4)
    paper_rgb = (min(255, max(220, p_base + tint)), p_base, min(255, max(218, p_base - tint)))
    shade = rng.randint(18, 65)
    ink_rgb = (shade, shade, min(255, shade + rng.randint(0, 35)))

    glyphs, cx = [], 24
    for ch in label:
        bb = font.getbbox(ch)
        cw = max(6, bb[2] - bb[0]) if ch != ' ' else max(8, int(fsize * 0.33))
        glyphs.append((ch, cx, rng.randint(-2, 2) if ch != ' ' else 0))
        cx += cw + rng.randint(-1, 2)

    ink = Image.new('L', (max(96, cx + 36), fsize * 3), 0)
    draw = ImageDraw.Draw(ink)
    sw = 1 if rng.random() < 0.22 else 0
    for ch, x, dy in glyphs:
        if ch != ' ':
            draw.text((x, fsize + dy), ch, fill=255, font=font, stroke_width=sw)

    shear = rng.uniform(-0.14, 0.14)
    if abs(shear) > 0.02:
        ink = ink.transform(ink.size, Image.Transform.AFFINE, (1.0, shear, -shear * fsize, 0.0, 1.0, 0.0), resample=Image.Resampling.BILINEAR)
    ang = rng.uniform(-1.8, 1.8)
    if abs(ang) > 0.2:
        ink = ink.rotate(ang, resample=Image.Resampling.BILINEAR, expand=True)

    arr = np.array(ink)
    ys, xs = np.where(arr > 28)
    x0, x1 = (max(0, int(xs.min()) - CROP_PADDING_PX), min(arr.shape[1], int(xs.max()) + CROP_PADDING_PX + 1)) if len(xs) else (0, 32)
    y0, y1 = (max(0, int(ys.min()) - CROP_PADDING_PX), min(arr.shape[0], int(ys.max()) + CROP_PADDING_PX + 1)) if len(ys) else (0, 24)
    alpha = arr[y0:y1, x0:x1].astype(np.float32) / 255.0
    h, w = alpha.shape
    paper = np.broadcast_to(np.array(paper_rgb, dtype=np.float32), (h, w, 3)).copy()
    paper += np.random.default_rng(rng.randint(0, 2**31 - 1)).normal(0.0, rng.uniform(1.2, 4.0), size=(h, w, 1))
    out = np.clip(paper * (1.0 - alpha[:, :, None]) + np.array(ink_rgb, dtype=np.float32) * alpha[:, :, None], 0, 255).astype(np.uint8)
    img = Image.fromarray(out, mode='RGB')
    blur = rng.uniform(0.0, 0.9)
    return img.filter(ImageFilter.GaussianBlur(radius=blur)) if blur > 0.25 else img

rng = random.Random(SEED)
all_fonts = sorted(FONTS_DIR.glob('*.ttf'), key=lambda p: p.name.lower())
accent_fonts = [p for p in all_fonts if font_supports_accents(p)]
fallback_accent = accent_fonts[0] if accent_fonts else None
rng.shuffle(all_fonts)
val_fonts = all_fonts[:3]     # Held-out handwriting styles never seen in training!
train_fonts = all_fonts[3:]

for split, count, fonts in (('train', SYNTH_TRAIN_COUNT, train_fonts), ('val', SYNTH_VAL_COUNT, val_fonts)):
    lines = []
    for i in range(count):
        kind, lbl = make_label(rng)
        im = render_crop(lbl, rng.choice(fonts), fallback_accent, rng)
        rel = f'{split}/{i:05d}_{kind}.png'
        im.save(DATASET_DIR / rel, format='PNG')
        lines.append(f'{rel}\t{lbl}')
    (DATASET_DIR / f'{split}_list.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')

# Optional merge if user uploaded real MyBizMate Scan *.zip exports into /content/exports
real_zips = sorted(Path('/content/exports').glob('*.zip')) if Path('/content/exports').exists() else []
if real_zips:
    print(f'Merging {len(real_zips)} real MyBizMate Scan export zip(s)...')
    real_idx = SYNTH_TRAIN_COUNT
    extra_lines = []
    for zp in real_zips:
        with zipfile.ZipFile(zp, 'r') as zf:
            manifest = json.loads(zf.read('manifest.json').decode('utf-8'))
            for scan in manifest.get('scans', []):
                for box in scan.get('boxes', []):
                    if not box.get('confirmed'):
                        continue
                    if not box.get('edited') and box.get('used') is not None:
                        continue
                    lbl = str(box.get('label', '')).replace('₱', 'P')
                    if 1 <= len(lbl) <= MAX_TEXT_LENGTH and all(c in ALLOWED_CHARS for c in lbl):
                        rel = f"train/{real_idx:05d}_real_{box['name']}.png"
                        (DATASET_DIR / rel).write_bytes(zf.read(box['crop']))
                        extra_lines.append(f'{rel}\t{lbl}')
                        real_idx += 1
    if extra_lines:
        with open(DATASET_DIR / 'train_list.txt', 'a', encoding='utf-8') as f:
            f.write('\n'.join(extra_lines) + '\n')
        print(f'Added {len(extra_lines)} verified real crops!')

print('Dataset ready at:', DATASET_DIR)
print('  Train fonts:', [f.name for f in train_fonts])
print('  Held-out Val fonts:', [f.name for f in val_fonts])

In [ ]:
# === CELL 4: DOWNLOAD OFFICIAL PP-OCRv6_small_rec & SLICE 18,710 -> 100 LATIN CLASSES ===
import numpy as np
import paddle

PRETRAIN_DIR = WORK_DIR / 'pretrain'
PRETRAIN_DIR.mkdir(exist_ok=True)

V6_WEIGHTS_URL = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_rec_pretrained.pdparams'
V6_WEIGHTS_PATH = PRETRAIN_DIR / 'PP-OCRv6_small_rec_pretrained.pdparams'
if not V6_WEIGHTS_PATH.exists():
    print('Downloading PP-OCRv6_small_rec_pretrained.pdparams (124.9 MB)...')
    urllib.request.urlretrieve(V6_WEIGHTS_URL, V6_WEIGHTS_PATH)

v6_dict_path = PADDLE_OCR_DIR / 'ppocr/utils/dict/ppocrv6_dict.txt'
v6_chars = v6_dict_path.read_text(encoding='utf-8').splitlines()
assert len(v6_chars) == 18708, f'Expected 18708 chars in ppocrv6_dict.txt, got {len(v6_chars)}'

src_char_to_zero_based = {ch: idx for idx, ch in enumerate(v6_chars)}
mapped_98 = [src_char_to_zero_based[ch] for ch in DICT_LINES]

# CTC class order: [0=<blank>, 1..98=dict.txt, 99=space (18709 in v6)]
ctc_indices = np.array([0] + [i + 1 for i in mapped_98] + [18709], dtype=np.int64)
# NRTR class order: [0=blank, 1=<unk>, 2=<s>, 3=</s>, 4..101=dict.txt, 102=space (18712 in v6)]
nrtr_103_indices = np.array([0, 1, 2, 3] + [i + 4 for i in mapped_98] + [18712], dtype=np.int64)
nrtr_104_indices = np.array([0, 1, 2, 3] + [i + 4 for i in mapped_98] + [18712, 18713], dtype=np.int64)

raw_state = paddle.load(str(V6_WEIGHTS_PATH))
state_dict = raw_state.get('state_dict', raw_state)
sliced_state = {}
sliced_layers = []

for key, tensor_val in state_dict.items():
    arr = np.asarray(tensor_val)
    old_shape = arr.shape
    for axis, dim in enumerate(old_shape):
        if dim == 18710:
            arr = np.take(arr, ctc_indices, axis=axis)
            sliced_layers.append((key, old_shape, arr.shape))
        elif dim == 18713:
            arr = np.take(arr, nrtr_103_indices, axis=axis)
            sliced_layers.append((key, old_shape, arr.shape))
        elif dim == 18714:
            arr = np.take(arr, nrtr_104_indices, axis=axis)
            sliced_layers.append((key, old_shape, arr.shape))
    sliced_state[key] = paddle.to_tensor(arr)

LATIN_INIT_PATH = PRETRAIN_DIR / 'PP-OCRv6_small_latin_cut_init.pdparams'
paddle.save(sliced_state, str(LATIN_INIT_PATH))
print('Sliced multilingual -> 100-class Latin head layers:')
for name, before, after in sliced_layers:
    print(f'  {name}: {before} -> {after}')

In [ ]:
# === CELL 5: WRITE FINE-TUNING CONFIG & MEASURE BASELINE ACCURACY BEFORE TRAINING ===
import yaml

base_cfg_path = PADDLE_OCR_DIR / 'configs/rec/PP-OCRv6/PP-OCRv6_small_rec.yml'
cfg = yaml.safe_load(base_cfg_path.read_text(encoding='utf-8'))

OUTPUT_DIR = WORK_DIR / 'output_ft'
cfg['Global']['pretrained_model'] = str(LATIN_INIT_PATH)
cfg['Global']['character_dict_path'] = str(DICT_PATH)
cfg['Global']['use_space_char'] = True
cfg['Global']['max_text_length'] = MAX_TEXT_LENGTH
cfg['Global']['save_model_dir'] = str(OUTPUT_DIR)
cfg['Global']['epoch_num'] = 15
cfg['Global']['save_epoch_step'] = 5
cfg['Global']['eval_batch_step'] = [0, 200]
cfg['Global']['distributed'] = False

# Low cosine learning rate preserves clean printed digits & accented glyphs while adapting to handwriting
cfg['Optimizer']['lr']['learning_rate'] = 0.0001
cfg['Optimizer']['lr']['warmup_epoch'] = 2

cfg['Train']['dataset']['data_dir'] = str(DATASET_DIR)
cfg['Train']['dataset']['label_file_list'] = [str(DATASET_DIR / 'train_list.txt')]
cfg['Train']['sampler']['first_bs'] = 64
cfg['Train']['loader']['batch_size_per_card'] = 64
cfg['Train']['loader']['num_workers'] = 2

cfg['Eval']['dataset']['data_dir'] = str(DATASET_DIR)
cfg['Eval']['dataset']['label_file_list'] = [str(DATASET_DIR / 'val_list.txt')]
cfg['Eval']['loader']['batch_size_per_card'] = 64
cfg['Eval']['loader']['num_workers'] = 2

FT_CFG_PATH = WORK_DIR / 'mybizmate_ppocrv6_small_latin.yml'
FT_CFG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')

print('Evaluating baseline pretrained accuracy on held-out validation fonts BEFORE fine-tuning...')
eval_before = subprocess.run(
    ['python', 'tools/eval.py', '-c', str(FT_CFG_PATH), '-o', f'Global.pretrained_model={LATIN_INIT_PATH}'],
    cwd=str(PADDLE_OCR_DIR), capture_output=True, text=True
)
for line in (eval_before.stdout + '\n' + eval_before.stderr).splitlines():
    if 'acc:' in line or 'norm_edit_dis:' in line:
        print('BASELINE ->', line)

In [ ]:
# === CELL 6: FINE-TUNE PP-OCRv6_small_rec ON GPU ===
!cd {PADDLE_OCR_DIR} && python tools/train.py -c {FT_CFG_PATH}

In [ ]:
# === CELL 7: EXPORT inference_latin.onnx, VERIFY C++ CONTRACT & DOWNLOAD ===
import hashlib
import json
import math
import cv2
import numpy as np
import onnx
import onnxruntime as ort

best_ckpt = OUTPUT_DIR / 'best_accuracy'
if not (OUTPUT_DIR / 'best_accuracy.pdparams').exists():
    best_ckpt = OUTPUT_DIR / 'latest'

INFER_EXPORT_DIR = WORK_DIR / 'infer_export'
if INFER_EXPORT_DIR.exists():
    shutil.rmtree(INFER_EXPORT_DIR)

subprocess.check_call([
    'python', 'tools/export_model.py',
    '-c', str(FT_CFG_PATH),
    '-o', f'Global.pretrained_model={best_ckpt}',
          f'Global.save_inference_dir={INFER_EXPORT_DIR}'
], cwd=str(PADDLE_OCR_DIR))

model_file = 'inference.json' if (INFER_EXPORT_DIR / 'inference.json').exists() else 'inference.pdmodel'
FINAL_ONNX_PATH = WORK_DIR / 'inference_latin.onnx'

subprocess.check_call([
    'paddle2onnx',
    '--model_dir', str(INFER_EXPORT_DIR),
    '--model_filename', model_file,
    '--params_filename', 'inference.pdiparams',
    '--save_file', str(FINAL_ONNX_PATH),
    '--opset_version', '11',
    '--enable_onnx_checker', 'True'
])

# Ensure single self-contained ONNX file (merge external tensors if any)
if Path(str(FINAL_ONNX_PATH) + '.data').exists():
    m = onnx.load(str(FINAL_ONNX_PATH), load_external_data=True)
    onnx.save_model(m, str(FINAL_ONNX_PATH), save_as_external_data=False)
    Path(str(FINAL_ONNX_PATH) + '.data').unlink(missing_ok=True)

# Strict C++ recognizer.cpp contract check
onnx_bytes = FINAL_ONNX_PATH.read_bytes()
sess = ort.InferenceSession(onnx_bytes, providers=['CPUExecutionProvider'])
in_name = sess.get_inputs()[0].name
out_name = sess.get_outputs()[0].name

for b_sz, width in ((1, 320), (8, 512)):
    dummy = np.random.uniform(-1.0, 1.0, size=(b_sz, 3, 48, width)).astype(np.float32)
    probs = sess.run([out_name], {in_name: dummy})[0]
    assert probs.ndim == 3 and probs.shape[0] == b_sz and probs.shape[2] == 100, f'Bad shape: {probs.shape}'
    assert float(probs.min()) >= -1e-5 and float(probs.max()) <= 1.0 + 1e-5, 'Not probabilities [0,1]'
    assert float(np.max(np.abs(probs.sum(axis=-1) - 1.0))) < 1e-3, 'Row sums != 1.0 (Softmax missing)'

sha256_hex = hashlib.sha256(onnx_bytes).hexdigest()
model_json = {
    'name': 'PP-OCRv6 small, Latin cut, ft-v1',
    'sha256': sha256_hex,
    'bytes': len(onnx_bytes),
}
MODEL_JSON_PATH = WORK_DIR / 'model.json'
MODEL_JSON_PATH.write_text(json.dumps(model_json, indent=2) + '\n', encoding='utf-8')

print('\n=== ALL C++ ONNX CONTRACT CHECKS PASSED! ===')
print('Saved:', FINAL_ONNX_PATH, f'({len(onnx_bytes):,} bytes)')
print('Copy this into modules/scanner/assets/scanner/model.json:')
print(json.dumps(model_json, indent=2))

try:
    from google.colab import files
    files.download(str(FINAL_ONNX_PATH))
    files.download(str(MODEL_JSON_PATH))
except ImportError:
    pass